# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [ ]:
# install Gurobi package in case not done yet:
%pip install gurobipy 
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
from pathlib import Path # for easier and robust folder and file handling across OS (Path can be used by Pandas directly)
import src.Shift as Shift # tailor-made data type for shift definitions
import src.functions as abd # self-made functions by Arty, Ben and Dirk... ;-) => call them by starting with "abd."


Note: you may need to restart the kernel to use updated packages.
c:\Users\dirkb\Documents\GitHub\Modellierungsseminar-Firestation\coding\src\functions.py
import pandas as pd
import datetime as dt
from pathlib import Path
import src.Shift as Shift # tailor-made data type for shift definitions


# LOGS / for process transparency 
def writeToLogs(yourStatusMessage:str, file, deleteHistory=False):
    try:
        if deleteHistory:
            with file.open("w") as log:
                log.write(dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                log.write(" // ")
                log.write(yourStatusMessage+"\n")
        else:
            with file.open("a") as log:
                log.write(dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                log.write(" // ")
                log.write(yourStatusMessage+"\n")
    except FileNotFoundError:
        print(f"file '{file}' not found - I skip logging and go on with my work...")
    except PermissionError:
        prin

### Read parameters

In [2]:
# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # main folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exorts of data sets for more transparency
FOLDER_OUTPUT = PROJECT_ROOT / "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs.txt"
abd.writeToLogs("cycle planning process started", FOLDER_AND_FILE_LOG, deleteHistory=True) # very first log entry deletes old log entries

In [3]:
# load parameters from CSV into dict
params = abd.readParameters(FOLDER_INPUT / "parameters.csv")
abd.writeToLogs("parameters loaded", FOLDER_AND_FILE_LOG)

### inputs and parameters

In [4]:
# global static variables

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(params["max_cycle_length"])   # max number of cycle weeks
MIN_REST        = int(params["min_rest"])            # min rest time between shifts in minutes
MAX_CONSEC_DAYS = int(params["max_conseq_working_days"])  # max consecutive working days

#MAX_CYCLE_WEEKS = 365  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
MAX_NB_CYLCEs = 3 # number of cycles

DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }
DICT_WEEKDAYS_RETURN = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}



### function readShiftSet

### function build_shift_objects

In [5]:

# basic inputs and parameters
cycles = range(MAX_NB_CYLCEs)            # 0 .. MAX_NB_CYLCEs-1
weeks  = range(MAX_CYCLE_WEEKS)          # 0 .. MAX_CYCLE_WEEKS-1
Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday

# improvements outstanding:
    # use files for parameter input
        # shift definitions => done
        # available staff => not required
        # user objectives: weighted priorities

creating shift objects:

In [6]:

# read input data for shift definitions
data_shiftSet = abd.readShiftSet(FOLDER_INPUT / "input_ShiftDataSet_Pesch.csv")

shift_objects = abd.build_shift_objects(data_shiftSet)
print(type(shift_objects))
#shift_objects = shift_objects.loc[shift_objects.index.repeat(shift_objects["required_staff"])] # multiply by required staff


WorkShifts = [s.shift_id for s in shift_objects] #object oriented solution
abd.writeToLogs(f"WorkShifts are defined as {WorkShifts}", FOLDER_AND_FILE_LOG)

freeDayShift = Shift.Shift("[freeDay]", 
                           "dummy shift for free days", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           0, 
                           1, 
                           0, 
                           False, 
                           None 
                           )
shift_objects.append(freeDayShift)

spareShift = Shift.Shift("[spareShift]", 
                           "dummy for spare shifts", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           1, 
                           5, 
                           0, 
                           True, 
                           None 
                           )
shift_objects.append(spareShift)

#log
abd.writeDataToLogs(shift_objects, FOLDER_LOGS / "log_shift_object.csv")

Shifts = [s.shift_id for s in shift_objects]
abd.writeToLogs(f"    Shifts are defined as {Shifts}", FOLDER_AND_FILE_LOG)

print(Shifts)


<class 'list'>
['[00day000week]_1', '[00day000week]_2', '[00day000week]_3', '[00day000week]_4', '[00day000week]_5', '[00day000week]_6', '[00day000week]_7', '[night000week]_1', '[night000week]_2', '[night000week]_3', '[night000week]_4', '[night000week]_5', '[00dayweekend]_1', '[00dayweekend]_2', '[00dayweekend]_3', '[00dayweekend]_4', '[00dayweekend]_5', '[nightweekend]_1', '[nightweekend]_2', '[nightweekend]_3', '[nightweekend]_4', '[nightweekend]_5', '[freeDay]', '[spareShift]']


### modelling

In [7]:
# modelling

modelCycle = gp.Model("SnakeBuilding_simple")

# --- new indices
# cycles = range(MAX_NB_CYLCEs)            # 0 .. MAX_NB_CYLCEs-1
# weeks  = range(MAX_CYCLE_WEEKS)          # 0 .. MAX_CYCLE_WEEKS-1

# --- decision variables
# x[c, s, d, sh] = 1 if in cycle c, cycleWeek s, on weekday d, shift sh is assigned
x = modelCycle.addVars(cycles, weeks, Weekdays, Shifts,
                       vtype=GRB.BINARY, name="x")

# active_cycle[c] = 1 if cycle c is used at all
active_cycle = modelCycle.addVars(cycles, vtype=GRB.BINARY, name="active_cycle")

# active_week[c,s] = 1 if cycle c uses cycleWeek s (i.e., at least one shift in that week is active)
active_week = modelCycle.addVars(cycles, weeks, vtype=GRB.BINARY, name="active_week")


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2806446
Academic license 2806446 - for non-commercial use only - registered to di___@student.uni-siegen.de


### conditions:

#### condition c01
_(idea is to use a unique ID for each condition for better reference)_

each shift has to be covered on each day

$$\sum_{s=1}^{n}{x_{s,d,w}} >= 1    \forall d \in D, \forall w \in W$$

$x_{s,d,w} = 1$, when snake s is working in work shift w on day d

$x: $ binary variable,
$s: $ snake number,
$d: $ weekday,
$ws: $ work shift


In [8]:
# enforce active_week >= any assignment in that week
for c in cycles:
    for s in weeks:
        modelCycle.addConstr(
            gp.quicksum(x[c, s, d, sh] for d in Weekdays for sh in Shifts) <=
            len(Weekdays) * len(Shifts) * active_week[c, s],
            name=f"Link_x_activeWeek_c{c}_s{s}_upper"
        )

# active_cycle must be 1 if any week in that cycle is active
for c in cycles:
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in weeks) <= MAX_CYCLE_WEEKS * active_cycle[c],
        name=f"Link_activeWeek_activeCycle_upper_c{c}"
    )
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in weeks) >= active_cycle[c],
        name=f"Link_activeWeek_activeCycle_lower_c{c}")

In [9]:
# SUBJECT TO:

# 1. each shift has to be covered on each day
for d in Weekdays:
    for ws in WorkShifts:
        modelCycle.addConstr(
            gp.quicksum(x[c, s, d, ws] for c in cycles for s in weeks) >= 1,
            name=f"Cover_day{d}_{ws}"
        )


####

#### condition c02

In [10]:
### DOUBLE-CHECK

# 2. each cycle shall have at most one shift per day
for c in cycles:
    for s in weeks:
        for d in Weekdays:
            modelCycle.addConstr(
                gp.quicksum(x[c, s, d, sh] for sh in Shifts) == active_week[c, s],
                name=f"OneShiftPerDay_c{c}_s{s}_d{d}"
            )


#### condition c03

In [11]:
### DOUBLE-CHECK

# 3) at max 5 consecutive working days (ensure time for resting)
for c in cycles:
    for s in weeks:
        for start in range(1, 8 - MAX_CONSEC_DAYS + 1):
            modelCycle.addConstr(
                gp.quicksum(x[c, s, d, sh] for d in range(start, start + MAX_CONSEC_DAYS) for sh in WorkShifts)
                <= MAX_CONSEC_DAYS,
                name=f"Max{MAX_CONSEC_DAYS}Work_c{c}_s{s}_start{start}"
            )



#### condition c04  

ensure that cycle weeks are activated in ascending order  

$$y_s - y_{s+1} >= 0$$

In [12]:
# 4) ensure that cycles and cycle weeks are activated in ascending order (not like 2-5-9-17-29-.... but 1-2-3-4-....)

for c in range(MAX_NB_CYLCEs - 1):
    modelCycle.addConstr(active_cycle[c] >= active_cycle[c + 1],
                         name=f"CycleOrder_c{c}")
for c in cycles:
    for s in range(MAX_CYCLE_WEEKS - 1):
        modelCycle.addConstr(active_week[c, s] >= active_week[c, s + 1],
                             name=f"WeekOrder_c{c}_s{s}")


### objective

In [13]:
# set objective function: minimize number of active snakes
modelCycle.setObjective(gp.quicksum(active_week[c, s] for c in cycles for s in range(MAX_CYCLE_WEEKS)), GRB.MINIMIZE)

# improvements outstanding:
    # add various weighted objectives => based on user input

#run optimizer
modelCycle.optimize()



Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 PRO 250 w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Academic license 2806446 - for non-commercial use only - registered to di___@student.uni-siegen.de
Optimize a model with 13299 rows, 185058 columns and 911044 nonzeros (Min)
Model fingerprint: 0x958e5535
Model has 1095 linear objective coefficients
Variable types: 0 continuous, 185058 integer (185058 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 4380 rows and 15330 columns
Presolve time: 1.04s
Presolved: 8919 rows, 169728 columns, 348763 nonzeros
Variable types: 0 continuous, 169728 integer (169728 binary)
Found heuristic solution: objective 22.0000000
Deterministic concurrent LP optimizer: prim

### results

In [14]:
# output (raw version, to be improved for better readability)
    # improvements outstanding:
        # write results in file
        # create a shift overview per staff member

if modelCycle.status == GRB.OPTIMAL:
    print("\nminimum number of cycle weeks:", int(modelCycle.objVal))
    abd.writeToLogs(f"successfully finished cycle plan: found an optimal solution using {int(modelCycle.objVal)} cycle weeks",FOLDER_AND_FILE_LOG)
    for c in range(MAX_NB_CYLCEs):
        if active_cycle[c].X == 1:
            print(f"\ncycle {c+1}:")
            abd.writeToLogs(f"\ncycle {c+1}:",FOLDER_AND_FILE_LOG)
            for s in range(MAX_CYCLE_WEEKS):
                if active_week[c, s].X == 1:
                    print(f"\ncycle week {s+1}:")
                    abd.writeToLogs(f"\ncycle week {s+1}:",FOLDER_AND_FILE_LOG)
                    for d in Weekdays:
                        for sh in Shifts:
                            if x[c, s, d, sh].X == 1:
                                print(f"\t{DICT_WEEKDAYS_RETURN[d]}: {sh}")
                                abd.writeToLogs(f"\tday {d}: {sh}",FOLDER_AND_FILE_LOG)


# output to file
if modelCycle.status == GRB.OPTIMAL:
    output_string = ""
    with open(FOLDER_OUTPUT / "output_cycle.csv", "w") as file:
        file.write("FINAL CYCLE:\nMon;Tue;Wed;Thu;Fri;Sat;Sun;\n")
    for c in range(MAX_NB_CYLCEs):
        if active_cycle[c].X == 1:
            for s in range(MAX_CYCLE_WEEKS):
                if active_week[c, s].X == 1:
                    for d in Weekdays:
                        for sh in Shifts:
                            if x[c, s, d, sh].X == 1:
                                output_string = output_string + sh + ";"
                    with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                        file.write(output_string + "\n")
                    output_string = ""




minimum number of cycle weeks: 22

cycle 1:

cycle week 1:
	Mon: [00day000week]_1
	Tue: [00day000week]_1
	Wed: [00day000week]_1
	Thu: [00day000week]_1
	Fri: [00day000week]_1
	Sat: [00day000week]_1
	Sun: [00day000week]_1

cycle week 2:
	Mon: [00day000week]_2
	Tue: [00day000week]_2
	Wed: [00day000week]_2
	Thu: [00day000week]_2
	Fri: [00day000week]_2
	Sat: [00day000week]_2
	Sun: [00day000week]_2

cycle week 3:
	Mon: [00day000week]_3
	Tue: [00day000week]_3
	Wed: [00day000week]_3
	Thu: [00day000week]_3
	Fri: [00day000week]_3
	Sat: [00day000week]_3
	Sun: [00day000week]_3

cycle week 4:
	Mon: [00day000week]_4
	Tue: [00day000week]_4
	Wed: [00day000week]_4
	Thu: [00day000week]_4
	Fri: [00day000week]_4
	Sat: [00day000week]_4
	Sun: [00day000week]_4

cycle week 5:
	Mon: [00day000week]_5
	Tue: [00day000week]_5
	Wed: [00day000week]_5
	Thu: [00day000week]_5
	Fri: [00day000week]_5
	Sat: [00day000week]_5
	Sun: [00day000week]_5

cycle week 6:
	Mon: [00day000week]_6
	Tue: [00day000week]_6
	Wed: [00day00

In [15]:
print(shift_objects[1])

Shift(shift_id='[00day000week]_2', description='day shift week', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri'], start=datetime.time(6, 0), end=datetime.time(16, 45), required_staff=7, shift_class=2, shift_work_time_assignment='nan', is_work_shift=True, required_qualification='none')
